# Search-Based Intelligence (SBI) - Kaggle bAbI Experiment Notebook

This notebook trains and evaluates SBI on the real English Facebook bAbI QA tasks from Kaggle: `roblexnana/the-babi-tasks-for-nlp-qa-system`.

Expected dataset size for tasks `[1, 2, 3]`: 3000 train examples and 3000 test examples.


In [ ]:
# Setup
!git clone https://github.com/karl4th/sbi.git
%cd sbi
!pip install -q -r requirements.txt


In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')
print(f'PyTorch: {torch.__version__}')


## 1. Download Kaggle bAbI Dataset

If Kaggle asks for credentials, upload `kaggle.json` to `~/.kaggle/kaggle.json` before running this cell.


In [ ]:
import os
from pathlib import Path
import kagglehub

path = kagglehub.dataset_download('roblexnana/the-babi-tasks-for-nlp-qa-system')
os.environ['BABI_DATA_DIR'] = path

root = Path(path)
en_files = sorted(root.rglob('en/qa*_train.txt'))
print('Path to dataset files:', path)
print('English train files:', len(en_files))
for p in en_files[:5]:
    print(' ', p.relative_to(root))
assert en_files, 'No English bAbI train files found under Kaggle dataset path'


## 2. Verify Real bAbI Parsing

This must print `Loaded real Kaggle bAbI`; if it falls back to synthetic data, stop and fix the dataset path.


In [ ]:
from data.babi.loader import load_babi_tasks, flatten_example

examples = load_babi_tasks(
    task_ids=[1, 2, 3],
    split='train',
    size_per_task=2,
    data_dir=os.environ['BABI_DATA_DIR'],
    allow_synthetic_fallback=False,
)

for ex in examples:
    inp, ans = flatten_example(ex)
    print(f'[Task {ex["task_id"]}]')
    print(f'  Input:  {inp[:160]}...')
    print(f'  Answer: {ans}')
    print()

print(f'Total examples loaded: {len(examples)}')


## 3. Check Dataset Sizes and Vocabulary


In [ ]:
from data.babi.dataset import BabiDataset

train_ds = BabiDataset(
    task_ids=[1, 2, 3],
    split='train',
    max_seq_len=256,
    data_dir=os.environ['BABI_DATA_DIR'],
    allow_synthetic_fallback=False,
)
test_ds = BabiDataset(
    task_ids=[1, 2, 3],
    split='test',
    tokenizer=train_ds.tokenizer,
    max_seq_len=256,
    data_dir=os.environ['BABI_DATA_DIR'],
    allow_synthetic_fallback=False,
)

print('Train:', len(train_ds))
print('Test :', len(test_ds))
print('Vocab:', train_ds.vocab_size)
assert len(train_ds) == 3000
assert len(test_ds) == 3000


## 4. Sanity Check SBI Components


In [ ]:
import sys
sys.path.insert(0, 'src')

import torch
from sbi.system import SBISystem
from sbi.core.config import SBIConfig, ReasoningConfig, MemoryConfig

cfg = SBIConfig(
    reasoning=ReasoningConfig(vocab_size=train_ds.vocab_size, n_layers=4, n_heads=4, d_model=256, d_ff=1024, hidden_dim=256),
    memory=MemoryConfig(fingerprint_dim=64, use_learned_fingerprint=False),
)
system = SBISystem(cfg)
print(f'Parameters: {system.num_parameters():,}')

dummy, _ = train_ds[0]
logits, fp = system(dummy.unsqueeze(0))
print(f'Logits: {logits.shape}, Fingerprint: {fp.shape}')
print('Sanity check passed')


## 5. Train Baseline


In [ ]:
!python3 training/train_baseline.py --config configs/baseline.yaml


## 6. Train SBI From Scratch


In [ ]:
!python3 training/train_sbi.py --config configs/sbi_small.yaml


## 7. Compare Results


In [ ]:
!python3 training/evaluate.py \
  --baseline experiments/checkpoints/baseline_best.pt \
  --sbi experiments/checkpoints/sbi_scratch_best.pt \
  --baseline-config configs/baseline.yaml \
  --sbi-config configs/sbi_small.yaml

from IPython.display import Image
Image('experiments/results/comparison.png')


## 8. Memory System Analysis


In [ ]:
import yaml
import torch
import matplotlib.pyplot as plt

from sbi.system import SBISystem
from sbi.core.config import SBIConfig, ReasoningConfig, MemoryConfig

with open('configs/sbi_small.yaml') as f:
    cfg_yaml = yaml.safe_load(f)

mc = cfg_yaml['model']
mm = cfg_yaml['memory']
sbi_config = SBIConfig(
    reasoning=ReasoningConfig(
        vocab_size=train_ds.vocab_size,
        max_seq_len=mc['max_seq_len'],
        n_layers=mc['n_layers'],
        n_heads=mc['n_heads'],
        d_model=mc['d_model'],
        d_ff=mc['d_ff'],
        dropout=mc['dropout'],
        hidden_dim=mc['d_model'],
    ),
    memory=MemoryConfig(
        fingerprint_dim=mm['fingerprint_dim'],
        max_memory_size=mm['max_memory_size'],
        hebbian_lr=mm['hebbian_lr'],
        decay_rate=mm['decay_rate'],
        min_confidence=mm['min_confidence'],
        compression_threshold=mm['compression_threshold'],
        top_k=mm['top_k'],
        use_learned_fingerprint=mm.get('use_learned_fingerprint', True),
    ),
)

sbi = SBISystem(sbi_config)
ckpt = torch.load('experiments/checkpoints/sbi_scratch_best.pt', map_location='cpu')
sbi.load_state_dict(ckpt['model_state'])

stats = sbi.memory_stats()
print(f"Memory entries : {stats['memory_size']}")
print(f"Hebbian edges  : {stats['graph_edges']}")
print(f"Training steps : {stats['step']}")

if sbi.episodic_memory.entries:
    confidences = [e.confidence for e in sbi.episodic_memory.entries]
    usage_counts = [e.usage_count for e in sbi.episodic_memory.entries]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].hist(confidences, bins=20)
    axes[0].set_title('Memory confidence')
    axes[1].hist(usage_counts, bins=20)
    axes[1].set_title('Memory usage count')
    plt.tight_layout()
    plt.show()


## 9. Reference Results

Published bAbI Memory Network results are near-perfect on tasks 1 and 2, while task 3 is harder. SBI should be compared against the baseline transformer on the same real Kaggle English bAbI split.
